####Join types in Apache Spark
 1. Inner Join - Return only the rows that have matching values on both sides
 2. Outer Joins
     * Left - Returns all rows from the left side and matching rows from the right side
     * Right - Returns all rows from the right side and matching rows from the left side
     * Full - Return all matching and non-matching rows from both sides
 3. Natural Join - Automatically create join criteria on the same column names (Applies to Inner and Outer Joins)
 4. Cross Join - Join without any join criteria (all possible combinations)
 5. Self Join - Join a table with itself (Applies to Inner, Outer, and Cross Joins)
 6. Semi Join - Take records from the left side when it matches with the right side
 7. Anti Join - Take records from the left side when it doesn not match with the right side
 8. Lateral Join - Each row from the driving table is used to subquery the derived table

####1. Load data from files and create the following tables.
 1. facilities -> facilities.csv
 2. members -> members.csv
 3. bookings -> bookings.csv

 Choose appropriate data types to best represent the data fields.

<br>

 <img src ='https://learningjournal.github.io/pub-resources/images/club_data_model.jpg' alt="Club Data Model" style="width: 300px">

####2. Define schema

In [0]:
facilities_schema = """facility_id INT, facility_name STRING, member_cost DOUBLE, guest_cost DOUBLE, 
    initial_outlay DOUBLE, monthly_maintainance DOUBLE"""

members_schema = """member_id INT, last_name STRING, first_name STRING, address STRING, zip_code STRING, 
    telephone STRING, recommended_by STRING, joining_date DATE"""

bookings_schema = "booking_id INT, facility_id INT, member_id INT, start_time TIMESTAMP, slots INT"

####3. Load facilities table

In [0]:
facilities_df = (
    spark.read.format("csv")
        .option("header", "true")
        .schema(facilities_schema)
        .load("/Volumes/dev/spark_db/datasets/spark_programming/data/facilities.csv")
)

facilities_df.write.mode("overwrite").saveAsTable("dev.spark_db.facilities")

####4. Load members table

In [0]:
members_df = (
    spark.read.format("csv")
        .option("header", "true")
        .schema(members_schema)
        .load("/Volumes/dev/spark_db/datasets/spark_programming/data/members.csv")
)

members_df.write.mode("overwrite").saveAsTable("dev.spark_db.members")

####5. Load bookings table

In [0]:
bookings_df = (
    spark.read.format("csv")
        .option("header", "true")
        .schema(bookings_schema)
        .load("/Volumes/dev/spark_db/datasets/spark_programming/data/bookings.csv")
)

bookings_df.write.mode("overwrite").saveAsTable("dev.spark_db.bookings")

###Inner Joins
----------------------------------------------------------------------------------------------------------------------------------------------

Q1. Prepare a facility bookings reporting dataset as the following.
 ```
 member_id | first_name | last_name | facility_id | slots | start_time
 ---------------------------------------------------------------------------
 ```
 The report must meet the following criteria.
 1. Facility bookings made by a person whose last name is Smith
 2. He has booked more than 5 slots in a single booking
 3. Report should be sorted by first name of the member in ascending order and number of slots in descending order

1.1 Try answering with SQL

 * Join Expression
 * Join type
 * Column name ambiguity

In [0]:
%sql

select m.member_id,m.first_name, m.last_name, b.facility_id, b.slots, b.start_time 
from dev.spark_db.members as m  inner join dev.spark_db.bookings as b on m.member_id=b.member_id
where m.last_name="Smith" and b.slots >5 
order by m.first_name asc, b.slots desc;

1.2 Try answering with dataframe api
 * Join Expression
 * Join type
 * Column name ambiguity


In [0]:
from pyspark.sql.functions import expr, col

members_df = spark.table("dev.spark_db.members").alias("m")
bookings_df = spark.table("dev.spark_db.bookings").alias("b")

#join_expr = expr("m.member_id == b.member_id")
join_expr = col("m.member_id") == col("b.member_id")
join_type = "inner"

reports_df = (
    members_df.join(bookings_df, join_expr, join_type)
    .filter("m.last_name='Smith' and b.slots >5 ")
    .select("m.member_id", "m.first_name", "m.last_name", "b.facility_id", "b.slots", "b.start_time")
    .orderBy("m.first_name", col("b.slots").desc())
)

reports_df.display()

Q2. Show me a facility bookings report as the following.
 ```
 member_id | first_name | last_name | facility_name | slots | booking_amount | start_time
 --------------------------------------------------------------------------------------------
 ```
 The report must meet the following criteria.
 1. Facility bookings made by a person whose last name is Smith
 2. He has booked more than 5 slots in a single booking
 3. Report should be sorted by first name of the member in ascending order and booking amount in descending order

In [0]:
from pyspark.sql.functions import col, expr

bookings_df = spark.table("dev.spark_db.bookings").alias("b")
members_df = spark.table("dev.spark_db.members").alias("m")
facilities_df = spark.table("dev.spark_db.facilities").alias("f")

report_df = (
    bookings_df
        .join(members_df, expr("b.member_id == m.member_id"), "inner")
        .join(facilities_df, col("b.facility_id") == col("f.facility_id"), "inner")
        .filter("m.last_name == 'Smith' and b.slots > 5")
        .selectExpr("m.member_id", "m.first_name", "m.last_name", "f.facility_name", "b.slots",
            "b.slots * f.member_cost as booking_amount", "b.start_time")
        .orderBy(col("m.first_name").asc(),
                 col("booking_amount").desc())
)
report_df.display()

###Outer Joins
----------------------------------------------------------------------------------------------------------------------------------------------

Q1. List all bookings made by a person named Darren Smith as the following.
 ```
 member_id | first_name | last_name | address | facility_id | slots
 ---------------------------------------------------------------------
 ```

 Ensure the following
 1. Show the details of all persons named Darren Smith even if they have not made any bookings
 2. Sort the result by number of slots (highers first)
 3. List the person with no bookings at the top

In [0]:
from pyspark.sql.functions import col, expr

bookings_df = spark.table("dev.spark_db.bookings").alias("b")
members_df = spark.table("dev.spark_db.members").alias("m")

join_expr = expr("m.member_id == b.member_id")
join_type = "left"

report_df = (
    members_df.join(bookings_df, join_expr, join_type)
    .filter("m.first_name == 'Darren' and m.last_name =='Smith'")
    .select("m.member_id", "m.first_name", "m.last_name", "b.facility_id", "b.slots")
    .orderBy(col("b.slots").desc_nulls_first())  #desc_nulls_first() puts nulls first , also we have desc_nulls_last() which will sort and put nulls in last
    )
report_df.display()

Q2. Show me a bookings report for Darren Smith as the following.
 ```
 facility_name | slots | booking_amount | start_time | member_id | member_name | telephone | address
 ------------------------------------------------------------------------------------------------------
 ```
 The report must meet the following criteria.

 1. Show the details of all persons named Darren Smith even if they have not made any bookings
 2. Sort the result by number of slots (highers first)
 3. List the person with no bookings at the top

In [0]:
from pyspark.sql.functions import expr, col

members_df = (
    spark.table("dev.spark_db.members")
        .where("first_name='Darren' and last_name='Smith'")
)

bookings_df = spark.table("dev.spark_db.bookings")
facilities_df = spark.table("dev.spark_db.facilities")

joined_df = (
    members_df.alias("m")
        .join(bookings_df.alias("b"), expr("m.member_id = b.member_id"), "left")
        .join(facilities_df.alias("f"), expr("b.facility_id=f.facility_id"), "left")
)

results_df = (
    joined_df.selectExpr(
        "f.facility_name", "b.slots",
        "b.slots * f.member_cost as booking_amount",
        "b.start_time", "b.member_id",
        "concat_ws(' ', m.first_name, m.last_name)  as member_name",
        "m.telephone", "m.address"
    ).orderBy(col("slots").desc_nulls_first())    
)

results_df.display()

Q3. Prepare a facility booking report as the following
 ```
 facility_name | member_cost | gest_cost | start_time | slots
 ---------------------------------------------------------------
 ```
 Ensure the following
 1. All club facilities must be listed in the report
 2. Consider only bookings for more than 10 slots

In [0]:
from pyspark.sql.functions import expr

facilities_df = spark.table("dev.spark_db.facilities")
bookings_df = spark.table("dev.spark_db.bookings").filter("slots > 10")

result_df = (
    bookings_df.join(facilities_df, bookings_df.facility_id == facilities_df.facility_id, "right")
             .select(facilities_df.facility_name,
                    facilities_df.member_cost,
                    facilities_df.guest_cost,
                    bookings_df.start_time,
                    bookings_df.slots)
)

result_df.display()

Q4. Prepare a member bookings report as the following
 ```
 booking_id | facility_name | slots | first_name | last_name | address
 ```
 Ensure the following
 1. Consider only regular memebrs (not guest) and direct members(not recomended by any other member)
 2. Consider only bookings for more than 8 hours
 3. Ensure all regular and direct members are listed even if they have no 8 hour bookings
 4. Ensure all 8 hour bookings are listed even if they are not made by regular and direct members
 5. Sort the report by slots and first name in ascending order

In [0]:
from pyspark.sql.functions import expr

members_df = (
    spark.table("dev.spark_db.members")
        .filter("member_id != 0 and recommended_by is null")
        .alias("m")
)

bookings_df = (
    spark.table("dev.spark_db.bookings")
        .filter("slots > 8")
        .alias("b")
)

facilities_df = spark.table("dev.spark_db.facilities").alias("f")

full_join_df = members_df.join(bookings_df, expr("m.member_id == b.member_id"), "full")

result_df = (
    full_join_df.join(facilities_df, expr("b.facility_id == f.facility_id"), "left")
    .select("b.booking_id","f.facility_name","b.slots","m.first_name","m.last_name","m.address")
    .orderBy(expr("b.slots").asc_nulls_last(), expr("m.first_name").asc_nulls_last())
)

display(result_df)

###Lateral Join
----------------------------------------------------------------------------------------------------------------------------------------------

####Lateral Join
 Lateral join allows to query right dataframe for each row of the left dataframe.\
 Lateral joins are especially useful when:
 1. You need per-parent Top-N child rows
 2. You want to invoke TVFs with arguments derived from each row

1. Find the most recent booking for each member
 ```
 +---------+----------+---------+---------------+-------------------+-----+
 |member_id|first_name|last_name|  facility_name|         start_time|slots|
 +---------+----------+---------+---------------+-------------------+-----+
 ```

In [0]:
from pyspark.sql.functions import col, expr

members_df = (
    spark.table("dev.spark_db.members")
        .filter("member_id > 0")
        .select("member_id", "first_name", "last_name")
        .alias("m")
)

bookings_df = spark.table("dev.spark_db.bookings").alias("b")
facilities_df = spark.table("dev.spark_db.facilities").alias("f")

latest_member_bookings_df = (
    # can use lateral join to query the right data frame. Here the right data frame was bookings query, the right data frame for each record of the left data frame members.

Here the right data frame was bookings query the right data frame for each record of the left data frame.
    members_df.lateralJoin(
        bookings_df.where("b.member_id == m.member_id")
            .orderBy(col("start_time").desc())
            .limit(1), None, "left")
        .select("m.member_id", "m.first_name", "m.last_name", "b.facility_id", "b.start_time", "b.slots")
).alias("mb")

result_df = (
    latest_member_bookings_df.join(facilities_df, on=expr("mb.facility_id == f.facility_id"), how="left")
    .select("mb.member_id", "mb.first_name", "mb.last_name", "f.facility_name", "mb.start_time", "mb.slots")
)

result_df.display()

2. Find all students with more than 1 years of Spark knowledge from offline_var_students

In [0]:
students_df = spark.table("dev.spark_db.offline_var_students").alias("s")

result_df = (
    #variant_explode is a table valued function which cannot be used in Select clause rather it can be used FROM clause, and thats where lateral joins comes into play. For each student record we want to explode the skills column.
    #tabled valued functions are not imported, they are registered in spark session which can be called
    #variant_explode returns a table of three columns Position, key and value, and the rows in the table is the elements of the array.
    students_df.lateralJoin(spark.tvf.variant_explode("s.skills")
                            .selectExpr("cast(value:Skill as string) as skill", "cast(value:YearsOfExperience as int) as experience"))
                .where("skill like '%Spark%' and experience > 1")
                .select("id", "first_name", "last_name", "skill", "experience")
)

result_df.display()

###Other Types of joins
1. Natural Join - Automatically create join criteria on the same column names (Applies to Inner and Outer Joins)

2. Cross Join - Join without any join criteria (all possible combinations): Cross join is a cross product of two data sets, in a join we have left data frame and right data frame and cross join allows us to do a cross product between these two left and right data frames. Cross product means match each record of the left data frame with each record of the right data frame.

3. Self Join - Join a table with itself (Applies to Inner, Outer, and Cross Joins): when the left side table and right side table are the same table.Then we call it Self-join.So it's just a logical name.There is no new type of join here.

4. Semi Join - Take records from the left side when it matches with the right side (Correlated EXISTS): Take records from the left side when it matches with the right side. It is like take record from the left data frame. If there is a matching record found in the right data frame.But do not take anything from the right data frame, take only things from the left data frame. Right data frame is only used to check the condition

5. Anti Join - Take records from the left side when it doesn not match with the right side (Correlated NOT EXISTS)

Q1. Show me a facility bookings report as the following. (Prefer Natural Join)
 ```
 member_id | first_name | last_name | facility_name | slots | booking_amount | start_time
 --------------------------------------------------------------------------------------------
 ```
 The report must meet the following criteria.
 1. Facility bookings made by a person whose last name is Smith
 2. He has booked more than 5 slots in a single booking
 3. Report should be sorted by first name of the member in ascending order and booking amount in descending order

In [0]:
from pyspark.sql.functions import col

bookings_df = spark.table("dev.spark_db.bookings").filter("slots > 5")
members_df = spark.table("dev.spark_db.members").filter("last_name == 'Smith'")
facilities_df = spark.table("dev.spark_db.facilities")

#In natural join we know that a field is similar in both the data frames, so we can just join the data frames without specifying the join expression such as b.member_id == m.member_id. In this case the field name is member_id which can be passed in alist, also if there are multiple columns which are similar in both the data frames, we can pass them in a list
#Natural join removes column ambiguity problem
bmf_df = bookings_df.join(members_df, ["member_id"], "inner").join(facilities_df, "facility_id")

report_df = (
    bmf_df.selectExpr("member_id", "first_name", "last_name", "facility_name", "slots",
                "slots * member_cost as booking_amount", "start_time")
        .orderBy("first_name", col("booking_amount").desc())
)

display(report_df)

Q2. Prepare a member bookings report as the following (Prefer Natural Join)
 ```
 booking_id | facility_name | slots | first_name | last_name | address
 ```
 Ensure the following
 1. Consider only regular memebrs (not guest) and direct members(not recomended by any other member)
 2. Consider only bookings for more than 8 hours
 3. Ensure all regular and direct members are listed even if they have no 8 hour bookings
 4. Ensure all 8 hour bookings are listed even if they are not made by regular and direct members
 5. Sort the report by slots and first name in ascending order

In [0]:
members_df = spark.table("dev.spark_db.members").filter("member_id != 0  and recommended_by is null")
bookings_df = spark.table("dev.spark_db.bookings").filter("slots > 8")
facilities_df = spark.table("dev.spark_db.facilities")

joined_df = (
    members_df.join(bookings_df, "member_id", "full")
            .join(facilities_df, "facility_id", "left")
)

report_df = (
    joined_df.select("booking_id", "facility_name", "slots", "first_name", "last_name", "address")
        .orderBy("slots", "first_name")
)

display(report_df)

Q3. How many bookings are possible when each member is booking a facility exactly once in a month?\
 Show all possible combinations (Cross Join combines left side of table with right side of the table and doesnt take any join expression)

In [0]:
members_df = spark.table("dev.spark_db.members").filter("member_id > 0")
facilities_df = spark.table("dev.spark_db.facilities")

report_df = (
    members_df.crossJoin(facilities_df)
        .select("first_name", "last_name", "facility_name")
)

report_df.display()

Q4. Prepare a report for members and who recomended them as the following(Self Join)
 ```
 member_id | Member Name | Recommended By
 --------------------------------------------
 ```

In [0]:
from pyspark.sql.functions import expr, concat_ws

members_df = spark.table("dev.spark_db.members")

report_df = (
    members_df.alias("m")
        .join(members_df.alias("r"), expr("m.recommended_by==r.member_id"), "inner")
        .select("m.member_id",
                concat_ws(" ", "m.first_name", "m.last_name").alias("Member Name"),
                concat_ws(" ", "r.first_name", "r.last_name").alias("Recommended By"))
)

report_df.display()

Q5. Prepare a list of members who made at least one booking. (Use SEMI Join)
 ```
 member_id | first_name | last_name | address
 -----------------------------------------------
 ```

In [0]:
members_df = spark.table("dev.spark_db.members").filter("member_id > 0").alias("m")
bookings_df = spark.table("dev.spark_db.bookings").alias("b")

report_df = (
    members_df.join(bookings_df, expr("m.member_id == b.member_id"), "left_semi")
        .select("member_id", "first_name", "last_name", "address")
)

report_df.display()

Q6. Prepare a list of members who never made any bookings. (Use ANTI Join)
 ```
 member_id | first_name | last_name | address
 -----------------------------------------------
 ```

In [0]:
members_df = spark.table("dev.spark_db.members").filter("member_id > 0").alias("m")
bookings_df = spark.table("dev.spark_db.bookings").alias("b")

report_df = (
    members_df.join(bookings_df, expr("m.member_id == b.member_id"), "left_anti")
        .select("member_id", "first_name", "last_name", "address")
)

report_df.display()